In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------
# FIXED DATASET PATH (your file)
# ---------------------------------------------------
DATA_PATH = "/content/drive/MyDrive/AML_Dataset/HI-Small_Trans.csv"

# Base directory where all Stage A outputs will be saved
BASE_DIR = "/content/drive/MyDrive/AML_Project_v2"
os.makedirs(BASE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print("DATA_PATH:", DATA_PATH)
print("BASE_DIR :", BASE_DIR)


DATA_PATH: /content/drive/MyDrive/AML_Dataset/HI-Small_Trans.csv
BASE_DIR : /content/drive/MyDrive/AML_Project_v2


In [ ]:
SAMPLE_CSV         = os.path.join(BASE_DIR, "HI_Trans_0p2.csv")
SCALER_PKL         = os.path.join(BASE_DIR, "ae_num_scaler_0p2.pkl")
PREPROC_CONFIG_JSON = os.path.join(BASE_DIR, "ae_preproc_config_0p2.json")
AE_MODEL_PATH      = os.path.join(BASE_DIR, "autoencoder_v2_0p2.pth")
AE_FEATURES_PKL    = os.path.join(BASE_DIR, "ae_v2_features_0p2.pkl")

print("Artifacts will be saved in:", BASE_DIR)


Artifacts will be saved in: /content/drive/MyDrive/AML_Project_v2


In [ ]:
df_full = pd.read_csv(DATA_PATH)
print("Full dataset shape:", df_full.shape)

df_02 = df_full.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
print("0.2 dataset shape:", df_02.shape)

df_02.to_csv(SAMPLE_CSV, index=False)
print("Saved sampled dataset →", SAMPLE_CSV)


Full dataset shape: (5078345, 11)
0.2 dataset shape: (1015669, 11)
Saved sampled dataset → /content/drive/MyDrive/AML_Project_v2/HI_Trans_0p2.csv


In [ ]:
# Target and account columns
LABEL_COL   = "Is Laundering"
ACCOUNT_COL = "Account"      # Source account, used later for GNN & fusion

# Transaction-level ID column for reference only
ID_COLS = ["Timestamp"]

# Base numeric columns
BASE_NUM_COLS = [
    "Amount Received",
    "Amount Paid",
]

# Low-cardinality categorical features for ONE-HOT encoding
CAT_ONEHOT_COLS = [
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
]

print("LABEL_COL      :", LABEL_COL)
print("ACCOUNT_COL    :", ACCOUNT_COL)
print("ID_COLS        :", ID_COLS)
print("BASE_NUM_COLS  :", BASE_NUM_COLS)
print("CAT_ONEHOT_COLS:", CAT_ONEHOT_COLS)

# Safety checks
assert LABEL_COL in df_02.columns
assert ACCOUNT_COL in df_02.columns
for c in BASE_NUM_COLS + CAT_ONEHOT_COLS:
    assert c in df_02.columns, f"Missing column: {c}"

print("Column configuration is VALID ✓")


LABEL_COL      : Is Laundering
ACCOUNT_COL    : Account
ID_COLS        : ['Timestamp']
BASE_NUM_COLS  : ['Amount Received', 'Amount Paid']
CAT_ONEHOT_COLS: ['Receiving Currency', 'Payment Currency', 'Payment Format']
Column configuration is VALID ✓


In [ ]:
# 1. Create engineered numeric features
df_02["Amount_Diff"]  = df_02["Amount Paid"] - df_02["Amount Received"]
df_02["Amount_Ratio"] = df_02["Amount Paid"] / (df_02["Amount Received"] + 1e-6)

NUM_COLS = BASE_NUM_COLS + ["Amount_Diff", "Amount_Ratio"]
print("Final numeric columns:", NUM_COLS)

# 2. Fit numeric scaler
num_scaler = StandardScaler()
X_num_scaled = num_scaler.fit_transform(df_02[NUM_COLS].astype(float))

# Save numeric scaler
joblib.dump(num_scaler, SCALER_PKL)
print("Saved numeric scaler →", SCALER_PKL)


Final numeric columns: ['Amount Received', 'Amount Paid', 'Amount_Diff', 'Amount_Ratio']
Saved numeric scaler → /content/drive/MyDrive/AML_Project_v2/ae_num_scaler_0p2.pkl


In [ ]:
# 3. One-hot encode low-cardinality categorical columns
df_cat = df_02[CAT_ONEHOT_COLS].astype(str)
df_cat_dummies = pd.get_dummies(df_cat, columns=CAT_ONEHOT_COLS, drop_first=False)
ONEHOT_COLS = df_cat_dummies.columns.tolist()

print("Number of one-hot columns:", len(ONEHOT_COLS))

# Save preprocessing config
preproc_config = {
    "label_col": LABEL_COL,
    "account_col": ACCOUNT_COL,
    "id_cols": ID_COLS,
    "num_cols": NUM_COLS,
    "onehot_cols": ONEHOT_COLS,
    "cat_onehot_base_cols": CAT_ONEHOT_COLS,
}

with open(PREPROC_CONFIG_JSON, "w") as f:
    json.dump(preproc_config, f, indent=2)

print("Saved preprocessing config →", PREPROC_CONFIG_JSON)


Number of one-hot columns: 37
Saved preprocessing config → /content/drive/MyDrive/AML_Project_v2/ae_preproc_config_0p2.json


In [ ]:
# Reload scaler and config to be extra sure
num_scaler = joblib.load(SCALER_PKL)
with open(PREPROC_CONFIG_JSON, "r") as f:
    preproc_config = json.load(f)

NUM_COLS         = preproc_config["num_cols"]
CAT_ONEHOT_COLS  = preproc_config["cat_onehot_base_cols"]
ONEHOT_COLS      = preproc_config["onehot_cols"]

# Numeric part
X_num = num_scaler.transform(df_02[NUM_COLS].astype(float))

# One-hot categorical part (must align columns with ONEHOT_COLS)
df_cat = df_02[CAT_ONEHOT_COLS].astype(str)
df_cat_dummies = pd.get_dummies(df_cat, columns=CAT_ONEHOT_COLS, drop_first=False)
df_cat_dummies = df_cat_dummies.reindex(columns=ONEHOT_COLS, fill_value=0)
X_cat = df_cat_dummies.values

# Final AE input matrix
X_ae = np.hstack([X_num, X_cat])
y = df_02[LABEL_COL].values

print("X_ae shape:", X_ae.shape)
print("Fraud count:", (y == 1).sum())


X_ae shape: (1015669, 41)
Fraud count: 1087


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_ae.shape[1]
latent_dim = 64   # increased latent size

class AEDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx]

class StrongAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

print("Using device:", device)
print("Input dim  :", input_dim)
print("Latent dim :", latent_dim)


Using device: cuda
Input dim  : 41
Latent dim : 64


In [ ]:
# Use only normal samples for AE training
normal_mask = (y == 0)
X_normal = X_ae[normal_mask]

X_train, X_val = train_test_split(
    X_normal, test_size=0.1, random_state=SEED
)

train_ds = AEDataset(X_train)
val_ds   = AEDataset(X_val)

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=1024, shuffle=False)

model = StrongAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
criterion = nn.MSELoss()

# Add small weight decay (L2 regularization)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

best_val_loss = float("inf")
epochs = 60   # can be slightly higher
noise_std = 0.05  # noise for denoising AE

for epoch in range(1, epochs + 1):
    # ---- Train ----
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)

        # Denoising: add Gaussian noise to inputs
        noisy_batch = batch + noise_std * torch.randn_like(batch)

        optimizer.zero_grad()
        x_hat, z = model(noisy_batch)
        loss = criterion(x_hat, batch)   # reconstruct original clean batch
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch.size(0)
    train_loss /= len(train_loader.dataset)

    # ---- Validate ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            x_hat, z = model(batch)   # no noise in validation
            loss = criterion(x_hat, batch)
            val_loss += loss.item() * batch.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), AE_MODEL_PATH)
        print("  → Saved best model")

print("Best validation loss:", best_val_loss)


Epoch 001 | train=0.103011 | val=0.138845
  → Saved best model
Epoch 002 | train=0.085035 | val=0.087164
  → Saved best model
Epoch 003 | train=0.069793 | val=0.066881
  → Saved best model
Epoch 004 | train=0.065725 | val=0.068990
Epoch 005 | train=0.055706 | val=0.072306
Epoch 006 | train=0.050320 | val=0.068876
Epoch 007 | train=0.046106 | val=0.112757
Epoch 008 | train=0.048811 | val=0.088065
Epoch 009 | train=0.050473 | val=0.065241
  → Saved best model
Epoch 010 | train=0.043137 | val=0.076301
Epoch 011 | train=0.034634 | val=0.075209
Epoch 012 | train=0.063694 | val=0.065068
  → Saved best model
Epoch 013 | train=0.043276 | val=0.069891
Epoch 014 | train=0.050742 | val=0.091833
Epoch 015 | train=0.035272 | val=0.076594
Epoch 016 | train=0.038978 | val=0.080297
Epoch 017 | train=0.034024 | val=0.090413
Epoch 018 | train=0.057051 | val=0.076399
Epoch 019 | train=0.032698 | val=0.072817
Epoch 020 | train=0.036838 | val=0.093114
Epoch 021 | train=0.033420 | val=0.076931
Epoch 022 | t

In [ ]:
# Reload best model
model = StrongAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
model.load_state_dict(torch.load(AE_MODEL_PATH, map_location=device))
model.eval()

full_ds = AEDataset(X_ae)
full_loader = DataLoader(full_ds, batch_size=1024, shuffle=False)

all_z = []
all_recon = []

with torch.no_grad():
    for batch in full_loader:
        batch = batch.to(device)
        x_hat, z = model(batch)
        mse = ((x_hat - batch) ** 2).mean(dim=1)  # per-sample recon error
        all_z.append(z.cpu().numpy())
        all_recon.append(mse.cpu().numpy())

all_z = np.concatenate(all_z, axis=0)
all_recon = np.concatenate(all_recon, axis=0)

print("Latent z shape    :", all_z.shape)
print("Recon error shape :", all_recon.shape)

# Build AE features DataFrame
z_cols = [f"AE_z_{i}" for i in range(all_z.shape[1])]
df_ae = pd.DataFrame(all_z, columns=z_cols)
df_ae["AE_recon_error"] = all_recon

# Add label, account, and IDs
df_ae[LABEL_COL]   = y
df_ae[ACCOUNT_COL] = df_02[ACCOUNT_COL].values
for col in ID_COLS:
    df_ae[col] = df_02[col].values

df_ae.to_pickle(AE_FEATURES_PKL)
print("Saved AE features →", AE_FEATURES_PKL)


Latent z shape    : (1015669, 64)
Recon error shape : (1015669,)
Saved AE features → /content/drive/MyDrive/AML_Project_v2/ae_v2_features_0p2.pkl


In [ ]:
print("===== STAGE A SUMMARY =====")

paths = {
    "Sampled 0.2 CSV"      : SAMPLE_CSV,
    "Scaler (num)"         : SCALER_PKL,
    "Preproc Config JSON"  : PREPROC_CONFIG_JSON,
    "AE Model Weights"     : AE_MODEL_PATH,
    "AE Features PKL"      : AE_FEATURES_PKL,
}

for name, path in paths.items():
    exists = os.path.exists(path)
    print(f"{name:25s} -> {'FOUND ✓' if exists else 'MISSING ✗'}  ({path})")

print("\nLoading AE features DF...")
df_ae = pd.read_pickle(AE_FEATURES_PKL)
print("AE features shape:", df_ae.shape)

print("\nAE feature columns:")
print(df_ae.columns.tolist())

print("\nSample AE feature rows:")
display(df_ae.head())

if LABEL_COL in df_ae.columns:
    print("\nLabel distribution (Is Laundering):")
    print(df_ae[LABEL_COL].value_counts())

if "AE_recon_error" in df_ae.columns:
    print("\nReconstruction error stats:")
    print(df_ae["AE_recon_error"].describe())


===== STAGE A SUMMARY =====
Sampled 0.2 CSV           -> FOUND ✓  (/content/drive/MyDrive/AML_Project_v2/HI_Trans_0p2.csv)
Scaler (num)              -> FOUND ✓  (/content/drive/MyDrive/AML_Project_v2/ae_num_scaler_0p2.pkl)
Preproc Config JSON       -> FOUND ✓  (/content/drive/MyDrive/AML_Project_v2/ae_preproc_config_0p2.json)
AE Model Weights          -> FOUND ✓  (/content/drive/MyDrive/AML_Project_v2/autoencoder_v2_0p2.pth)
AE Features PKL           -> FOUND ✓  (/content/drive/MyDrive/AML_Project_v2/ae_v2_features_0p2.pkl)

Loading AE features DF...
AE features shape: (1015669, 68)

AE feature columns:
['AE_z_0', 'AE_z_1', 'AE_z_2', 'AE_z_3', 'AE_z_4', 'AE_z_5', 'AE_z_6', 'AE_z_7', 'AE_z_8', 'AE_z_9', 'AE_z_10', 'AE_z_11', 'AE_z_12', 'AE_z_13', 'AE_z_14', 'AE_z_15', 'AE_z_16', 'AE_z_17', 'AE_z_18', 'AE_z_19', 'AE_z_20', 'AE_z_21', 'AE_z_22', 'AE_z_23', 'AE_z_24', 'AE_z_25', 'AE_z_26', 'AE_z_27', 'AE_z_28', 'AE_z_29', 'AE_z_30', 'AE_z_31', 'AE_z_32', 'AE_z_33', 'AE_z_34', 'AE_z_35', 'A

,AE_z_0,AE_z_1,AE_z_2,AE_z_3,AE_z_4,AE_z_5,AE_z_6,AE_z_7,AE_z_8,AE_z_9,...,AE_z_58,AE_z_59,AE_z_60,AE_z_61,AE_z_62,AE_z_63,AE_recon_error,Is Laundering,Account,Timestamp
0,-0.171639,-4.232361,0.104417,-3.298181,4.128160,1.148903,-4.008280,0.183923,4.746537,-1.055284,...,2.734672,-0.448340,-0.755428,-3.457057,-0.710143,1.532658,0.000016,0,80E50C3C0,2022/09/01 00:29
1,-0.552513,1.176918,0.085590,1.292784,2.337363,3.379844,-0.583980,-1.723102,0.499747,2.023540,...,1.297836,-0.107377,1.208097,-3.487659,-2.267819,0.238896,0.000018,0,8001C6CC0,2022/09/01 13:28
2,0.693883,2.977150,0.714605,1.302704,5.107666,1.099602,-2.010736,0.382961,6.933664,1.577050,...,2.689955,-2.235557,-1.524162,-6.733779,-1.529114,-3.113523,0.000078,0,80CAF3CE0,2022/09/01 02:46
3,-2.277668,1.706605,1.744195,-0.888375,2.416312,-2.294415,-2.257886,0.131887,5.121434,1.695682,...,-0.526143,1.533264,-0.255519,-5.008838,-2.726768,1.109005,0.000049,0,804DC2C20,2022/09/02 08:02
4,-0.396464,-0.740129,-2.468286,0.665525,1.697380,3.444777,-0.233828,-2.433388,0.482001,0.393666,...,4.090092,0.048419,-0.156657,-2.817705,-2.929527,0.502151,0.000062,0,80A5EC8A0,2022/09/09 18:01



Label distribution (Is Laundering):
Is Laundering
0    1014582
1       1087
Name: count, dtype: int64

Reconstruction error stats:
count    1.015669e+06
mean     6.440570e-03
std      2.762207e+00
min      6.674582e-06
25%      1.784611e-05
50%      3.049974e-05
75%      8.887213e-05
max      2.281948e+03
Name: AE_recon_error, dtype: float64


In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

print("===== AUTOENCODER EVALUATION METRICS =====")

df_ae = pd.read_pickle(AE_FEATURES_PKL)

re = df_ae["AE_recon_error"].values
labels = df_ae[LABEL_COL].values

# 1️⃣ Basic stats
mean_normal = re[labels == 0].mean()
mean_fraud  = re[labels == 1].mean()

print(f"Mean recon error (Normal) : {mean_normal:.6f}")
print(f"Mean recon error (Fraud)  : {mean_fraud:.6f}")
print(f"Difference                : {mean_fraud - mean_normal:.6f}")

# 2️⃣ ROC-AUC
auc = roc_auc_score(labels, re)
print(f"\nROC-AUC (reconstruction error): {auc:.4f}")

# 3️⃣ Threshold scan for best F1
thresholds = np.linspace(re.min(), re.max(), 200)
best_thresh, best_f1 = None, 0.0

for t in thresholds:
    y_pred = (re > t).astype(int)
    f1 = f1_score(labels, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\nBest Threshold (F1 optimized): {best_thresh:.6f}")
print(f"Best F1 Score: {best_f1:.4f}")

# 4️⃣ Final metrics at best threshold
y_pred_best = (re > best_thresh).astype(int)
precision = precision_score(labels, y_pred_best)
recall    = recall_score(labels, y_pred_best)
f1        = f1_score(labels, y_pred_best)

print("\n===== FINAL METRICS AT BEST THRESHOLD =====")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


===== AUTOENCODER EVALUATION METRICS =====
Mean recon error (Normal) : 0.006447
Mean recon error (Fraud)  : 0.000088
Difference                : -0.006359

ROC-AUC (reconstruction error): 0.6407

Best Threshold (F1 optimized): 0.000007
Best F1 Score: 0.0021

===== FINAL METRICS AT BEST THRESHOLD =====
Precision: 0.0011
Recall   : 1.0000
F1 Score : 0.0021
